In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def generate_iaaft_surrogate(signal, max_iter=1000):
    """
    Mô phỏng thuật toán IAAFT (Iterative Amplitude Adjusted Fourier Transform).
    
    Args:
        signal (array-like): Chuỗi thời gian gốc.
        max_iter (int): Số vòng lặp tối đa để tránh lặp vô hạn.
        
    Returns:
        tuple: (chuỗi Surrogate IAAFT, số vòng lặp đã thực hiện)
    """
    # =================================================================
    # BƯỚC 0: Khởi tạo (Initialization)
    # =================================================================
    signal = np.asarray(signal).flatten()
    n = len(signal)
    
    # Lưu phân phối gốc (Mảng đã sắp xếp)
    s_sorted = np.sort(signal)
    
    # Lưu biên độ phổ Fourier lý tưởng
    target_amplitudes = np.abs(np.fft.rfft(signal))
    
    # Khởi tạo chuỗi giả bằng cách xáo trộn ngẫu nhiên dữ liệu gốc
    s_curr = np.random.permutation(signal)
    
    # =================================================================
    # VÒNG LẶP ĐỆ QUY
    # =================================================================
    for i in range(max_iter):
        # BƯỚC 1: Khớp Phổ năng lượng (The Fourier Step)
        # Chuyển sang miền tần số
        S_curr = np.fft.rfft(s_curr)
        phases = np.angle(S_curr)
        
        # Cưỡng ép phổ: Lấy biên độ gốc ghép với góc pha hiện tại
        S_new = target_amplitudes * np.exp(1j * phases)
        
        # Biến đổi ngược về miền thời gian (tạo chuỗi trung gian)
        s_tilde = np.fft.irfft(S_new, n=n)
        
        # BƯỚC 2: Khớp Phân phối (The Rank-ordering Step)
        # Lấy thứ hạng của chuỗi trung gian
        ranks = np.argsort(np.argsort(s_tilde))
        
        # Cưỡng ép phân phối: Áp giá trị gốc vào thứ hạng mới
        s_next = s_sorted[ranks]
        
        # BƯỚC 3: Kiểm tra Hội tụ (Convergence)
        # Nếu không có phần tử nào bị hoán đổi vị trí nữa, thuật toán dừng
        if np.array_equal(s_next, s_curr):
            return s_next, i + 1
            
        s_curr = s_next
        
    # Trả về kết quả nếu chạm ngưỡng max_iter (hội tụ tiệm cận)
    return s_curr, max_iter

# =====================================================================
# THIẾT LẬP DỮ LIỆU VÀ CHẠY MÔ PHỎNG
# =====================================================================
if __name__ == "__main__":
    np.random.seed(42)
    N = 1000
    t = np.arange(N)
    
    # Tạo tín hiệu gốc phi chuẩn mạnh để thử thách thuật toán
    x_orig = np.exp(np.sin(2 * np.pi * t / 50)) + 0.2 * np.random.randn(N)
    
    # 1. Chạy IAAFT
    x_iaaft, iterations = generate_iaaft_surrogate(x_orig)
    print(f"IAAFT hội tụ thành công sau {iterations} vòng lặp.")
    
    # 2. Tạo một chuỗi AAFT cũ (chỉ lặp đúng 1 lần) để so sánh
    x_aaft = np.sort(x_orig)[np.argsort(np.argsort(np.fft.irfft(
        np.abs(np.fft.rfft(np.random.permutation(x_orig))) * 
        np.exp(1j * np.random.uniform(0, 2*np.pi, len(np.fft.rfft(x_orig)))), n=N)))]
        
    # =================================================================
    # TÍNH TOÁN CÁC ĐẶC TRƯNG ĐỂ SO SÁNH
    # =================================================================
    freqs = np.fft.rfftfreq(N)
    psd_orig = np.abs(np.fft.rfft(x_orig))**2
    psd_aaft = np.abs(np.fft.rfft(x_aaft))**2
    psd_iaaft = np.abs(np.fft.rfft(x_iaaft))**2
    
    def autocorrelation(signal):
        centered = signal - np.mean(signal)
        autocorr = np.correlate(centered, centered, mode='full')
        return (autocorr[len(autocorr)//2:] / autocorr[len(autocorr)//2])[:150]
        
    ac_orig = autocorrelation(x_orig)
    ac_aaft = autocorrelation(x_aaft)
    ac_iaaft = autocorrelation(x_iaaft)
    
    # =================================================================
    # TRỰC QUAN HÓA BẰNG MATPLOTLIB
    # =================================================================
    plt.figure(figsize=(16, 10))
    
    # Đồ thị 1: Chuỗi thời gian
    plt.subplot(2, 2, 1)
    plt.plot(t[:300], x_orig[:300], label='x gốc', color='blue')
    plt.plot(t[:300], x_iaaft[:300], label=f'IAAFT (Đã hội tụ)', color='green', alpha=0.8)
    plt.title('Chuỗi thời gian: IAAFT phá vỡ góc pha phi tuyến')
    plt.legend()
    
    # Đồ thị 2: Histogram (Chứng minh biên độ khớp 100%)
    plt.subplot(2, 2, 2)
    plt.hist(x_orig, bins=40, alpha=0.5, color='blue', density=True, label='x gốc')
    plt.hist(x_iaaft, bins=40, alpha=0.5, color='green', density=True, label='IAAFT')
    plt.title('BƯỚC 2: Phân phối biên độ khớp tuyệt đối 100%')
    plt.legend()
    
    # Đồ thị 3: Phổ năng lượng (Sự vượt trội của IAAFT so với AAFT)
    plt.subplot(2, 2, 3)
    plt.plot(freqs, psd_orig, label='PSD Gốc', color='blue', linewidth=2)
    plt.plot(freqs, psd_aaft, label='PSD AAFT (Bị hụt)', color='red', linestyle='--')
    plt.plot(freqs, psd_iaaft, label='PSD IAAFT (Khôi phục)', color='green', linestyle='-.')
    plt.xlim(0, 0.1)
    plt.title('BƯỚC 1: Phổ năng lượng được vá lỗi nhờ vòng lặp')
    plt.legend()
    
    # Đồ thị 4: Tự tương quan
    plt.subplot(2, 2, 4)
    plt.plot(ac_orig, label='Tương quan Gốc', color='blue', linewidth=2)
    plt.plot(ac_aaft, label='Tương quan AAFT (Bị yếu)', color='red', linestyle='--')
    plt.plot(ac_iaaft, label='Tương quan IAAFT (Khôi phục)', color='green', linestyle='-.')
    plt.title('Hàm tự tương quan: Cấu trúc "trí nhớ" tuyến tính được phục hồi')
    plt.legend()
    
    plt.tight_layout()
    plt.show()